This notebook identifies and removes rows from the filtered CoT datasets whose pre-verdict extraction lands on a lexically confounded word — i.e. a final token that, within a given task, correlates strongly with the label regardless of the model's actual truth representation (e.g. `doesn't`/`isn't`-type negation auxiliaries).

Workflow:
1. Print the overall (pooled across all tasks) landing-word distribution, for reference.
2. Print the per-task landing-word distribution — this is the one that actually matters, since pooled numbers can hide task-specific skew (words can cancel out across tasks while still being confounds within any single task).
3. Manually specify, per task, which landing words to treat as confounds based on the printed tables.
4. Drop those rows from the filtered CSVs (in place) and report how many were removed per task.

This notebook does NOT build the lexical-shortcut control baseline (the one-hot-landing-word logistic regression used to sanity-check reported AUROC numbers) — that's an analysis artifact, not a dataset-generation step, and belongs in `experiments/` instead.


In [1]:
import re
from pathlib import Path

import pandas as pd

FILTERED_DIR = Path("../CoT_datasets/filtered")
RAW_LABELS_DIR = Path("../dataset")  # has the ground-truth `label` column; filtered CoT files don't
SPLITS = ["train", "test"]

# discover tasks dynamically from whatever's actually been filtered so far
TASKS = sorted({p.stem.split("_filtered_")[0] for p in FILTERED_DIR.glob("*_filtered_train.csv")})
print("Tasks found:", TASKS)

# Utility Functions
def get_statement(generated_text: str) -> str:
    return generated_text.split("<｜User｜>")[1].split("\n")[0].strip()


def final_word(text: str):
    m = re.search(r"([A-Za-z']+)\s*$", text.rstrip())
    return m.group(1).lower() if m else None


def load_task(task: str, split: str) -> pd.DataFrame:
    filt = pd.read_csv(FILTERED_DIR / f"{task}_filtered_{split}.csv")
    raw = pd.read_csv(RAW_LABELS_DIR / f"{task}_{split}.csv")[["statement", "label"]].drop_duplicates(subset="statement")

    filt = filt.copy()
    filt["statement"] = filt["generated_statement_texts"].apply(get_statement)
    merged = filt.merge(raw, on="statement", how="left")

    unmatched = merged["label"].isna().sum()
    if unmatched:
        print(f"  WARNING {task}_{split}: {unmatched}/{len(merged)} rows failed to recover a label")

    merged["final_word"] = merged["extracted_statement_texts"].apply(final_word)
    merged["task"] = task
    merged["split"] = split
    return merged


all_rows = pd.concat(
    [load_task(task, "train") for task in TASKS],
    ignore_index=True,
).dropna(subset=["label"])
all_rows["label"] = all_rows["label"].astype(bool)

print(f"\nTotal rows loaded across all tasks/splits: {len(all_rows)}")

Tasks found: ['A1', 'A2', 'A3', 'F0', 'F1', 'F2', 'F3', 'F4', 'F5']

Total rows loaded across all tasks/splits: 9834


## 1. Overall landing-word distribution (pooled across all tasks)

Reference only — pooled numbers can look neutral even when a word is a severe confound within a single task, since opposite-direction skews across tasks can cancel out. Don't decide drops from this table alone; see the per-task breakdown below.

In [2]:
MIN_COUNT = 10  # ignore landing words that barely occur; too few rows to judge skew meaningfully


def print_distribution(df: pd.DataFrame, min_count: int = MIN_COUNT):
    counts = df["final_word"].value_counts()
    counts = counts[counts >= min_count]
    print(f"{'word':15s}{'n':>7s}{'true':>7s}{'false':>7s}{'%true':>8s}{'|dev from 50|':>15s}")
    stats = []
    for w, n in counts.items():
        sub = df[df["final_word"] == w]
        t = int(sub["label"].sum())
        pct = 100 * t / n
        stats.append((w, n, t, n - t, pct, abs(pct - 50)))
    stats.sort(key=lambda r: r[5])  # most neutral first, most skewed last
    for w, n, t, f, pct, dev in stats:
        print(f"{w!r:15s}{n:7d}{t:7d}{f:7d}{pct:7.1f}%{dev:14.1f}pt")


print_distribution(all_rows)

word                 n   true  false   %true  |dev from 50|
'actually'          22     11     11   50.0%           0.0pt
'be'              2812   1289   1523   45.8%           4.2pt
'being'             42     18     24   42.9%           7.1pt
'statement'        435    249    186   57.2%           7.2pt
'definitely'        19      8     11   42.1%           7.9pt
'is'              2335    924   1411   39.6%          10.4pt
'seem'              86     53     33   61.6%          11.6pt
'seems'            236    146     90   61.9%          11.9pt
'likely'            22      7     15   31.8%          18.2pt
'it'                20     14      6   70.0%          20.0pt
"it's"              15     11      4   73.3%          23.3pt
'also'              14      3     11   21.4%          28.6pt
'which'            615    497    118   80.8%          30.8pt
'are'              380    320     60   84.2%          34.2pt
"that's"           369    311     58   84.3%          34.3pt
'that'             407   

## 2. Per-task landing-word distribution

This is the table that actually matters for deciding what to drop — the same word can be near-neutral in one task and severely skewed in another.

In [3]:
for task in TASKS:
    print(f"\n{'=' * 60}\n{task}\n{'=' * 60}")
    print_distribution(all_rows[all_rows["task"] == task], min_count=MIN_COUNT)


A1
word                 n   true  false   %true  |dev from 50|
'is'               123     72     51   58.5%           8.5pt
'be'               136     51     85   37.5%          12.5pt
'seem'              12      4      8   33.3%          16.7pt
"that's"            10      7      3   70.0%          20.0pt
'that'             108     92     16   85.2%          35.2pt
'seems'             20     19      1   95.0%          45.0pt
"there's"           28      1     27    3.6%          46.4pt
"doesn't"          132      1    131    0.8%          49.2pt
'checks'            20     20      0  100.0%          50.0pt
'holds'             17     17      0  100.0%          50.0pt
"isn't"             12      0     12    0.0%          50.0pt

A2
word                 n   true  false   %true  |dev from 50|
'be'                46     24     22   52.2%           2.2pt
'is'                78     47     31   60.3%          10.3pt
'that'              55     46      9   83.6%          33.6pt
"that's"          

## 3. Final procedure: shift perfectly-monotone words, then balance every word to min(True, False)

Two steps, applied independently to each task+split (each split uses only its own labels — no
train/test leakage concern here, since there's no cross-split decision being made, just the
same procedure run separately on each split's own data):

1. **Shift perfectly-monotone landing words** (0% or 100% true, e.g. `doesn't`/`there's`) back
   one word. Provably safe regardless of count: a monotone word always contributes zero rows
   under step 2 (`min(true, false) = min(0, n) = 0`), so merging it into whatever word precedes
   it can only add representation, never lose any — verified empirically across all tasks
   (delta was never negative). We deliberately do NOT extend this to near-monotone words (e.g.
   `holds` at 95.9%) — that was tested and found to not be guaranteed safe (can net-lose rows
   if the shifted rows land on a similarly-skewed target), and the gain from doing so was small
   relative to the risk.
2. **Balance every remaining landing word to `min(true_count, false_count)`** — no threshold,
   no manual word list, applied uniformly to every word including ones that look "fine," so no
   single landing word can ever predict the label better than chance.

Saves the result into `CoT_datasets/lexically_cleaned/`, alongside the original `raw/` and
`filtered/` dirs — leaves `CoT_datasets/filtered/` untouched.</cell id="6dacc63e">


In [4]:
from ast import literal_eval
import zlib

import numpy as np
from transformers import AutoTokenizer

from filtering_utils import _char_to_token_idx

RANDOM_SEED = 42
LEXICALLY_CLEANED_DIR = Path("../CoT_datasets/lexically_cleaned")
LEXICALLY_CLEANED_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")


def last_two_words(text: str):
    words = re.findall(r"[A-Za-z']+", text.rstrip())
    if len(words) < 2:
        return None, (words[-1].lower() if words else None)
    return words[-2].lower(), words[-1].lower()


def reextract_after_shift(row, tokenizer):
    """Actually re-truncates extracted_statement_ids/texts to end at the shifted-to word,
    not just relabel it for bucketing -- otherwise the shift has zero effect on what the
    probe actually reads."""
    text = row["extracted_statement_texts"].rstrip()
    m = re.search(r"[A-Za-z']+\s*$", text)
    if m is None:
        return row["extracted_statement_ids"], row["extracted_statement_texts"]

    new_text_prefix = text[:m.start()].rstrip()
    target_char = len(new_text_prefix)

    ids = row["generated_statement_ids"]
    token_idx = _char_to_token_idx(tokenizer=tokenizer, ids=ids, upper_tok=len(ids), target_char=target_char)
    new_ids = ids[:token_idx]
    new_text = tokenizer.decode(new_ids, skip_special_tokens=True)
    return str(new_ids), new_text


def clean_dataset(task: str, split: str) -> pd.DataFrame:
    # independent per-task+split seed -- regenerating any one task/split can never change
    # another's row selection, unlike a single RNG shared/advanced across the whole loop
    rng = np.random.RandomState(RANDOM_SEED + zlib.crc32(f"{task}_{split}".encode()))

    path = FILTERED_DIR / f"{task}_filtered_{split}.csv"
    raw = pd.read_csv(RAW_LABELS_DIR / f"{task}_{split}.csv")[["statement", "label"]].drop_duplicates(subset="statement")

    df = pd.read_csv(path)
    if isinstance(df["generated_statement_ids"].iloc[0], str):
        df["generated_statement_ids"] = df["generated_statement_ids"].apply(literal_eval)

    df["statement"] = df["generated_statement_texts"].apply(get_statement)
    df = df.merge(raw, on="statement", how="left")

    unmatched = df["label"].isna().sum()
    if unmatched:
        print(f"  {task}_{split}: {unmatched} rows failed to recover a label, dropping them")
        df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(bool)

    df[["_prev_word", "_last_word"]] = df["extracted_statement_texts"].apply(
        lambda t: pd.Series(last_two_words(t))
    )

    # step 1: shift perfectly-monotone words (any count -- always safe, see markdown above)
    monotone_words = set()
    for w, group in df.groupby("_last_word", dropna=True):
        t = group["label"].sum()
        if t == 0 or t == len(group):
            monotone_words.add(w)

    shift_mask = df["_last_word"].isin(monotone_words)
    df["_final_word"] = df["_prev_word"].where(shift_mask, df["_last_word"])

    if shift_mask.any():
        reextracted = df.loc[shift_mask].apply(lambda r: reextract_after_shift(r, tokenizer), axis=1)
        df.loc[shift_mask, "extracted_statement_ids"] = reextracted.apply(lambda p: p[0])
        df.loc[shift_mask, "extracted_statement_texts"] = reextracted.apply(lambda p: p[1])
        print(f"  {task}_{split}: re-truncated {shift_mask.sum()} shifted rows (words: {sorted(monotone_words)})")

    # step 2: balance every word to min(true, false)
    keep_parts = []
    for w, group in df.groupby("_final_word", dropna=False):
        true_rows = group[group["label"]]
        false_rows = group[~group["label"]]
        n = min(len(true_rows), len(false_rows))
        if n == 0:
            continue
        keep_parts.append(true_rows.sample(n=n, random_state=rng))
        keep_parts.append(false_rows.sample(n=n, random_state=rng))

    cleaned = pd.concat(keep_parts).sort_index()
    cleaned["generated_statement_ids"] = cleaned["generated_statement_ids"].apply(str)
    return cleaned.drop(columns=["statement", "_prev_word", "_last_word", "_final_word"])

In [5]:
total_before = total_after = 0
for task in TASKS:
    for split in SPLITS:
        n_before = len(pd.read_csv(FILTERED_DIR / f"{task}_filtered_{split}.csv"))
        cleaned = clean_dataset(task, split)
        out_path = LEXICALLY_CLEANED_DIR / f"{task}_{split}.csv"
        cleaned.to_csv(out_path, index=False)

        n_after = len(cleaned)
        total_before += n_before
        total_after += n_after
        print(f"{task}_{split}: {n_before} -> {n_after} rows ({100*n_after/n_before:.1f}%) -> {out_path}")

print(f"\nOVERALL: {total_before} -> {total_after} rows ({100*total_after/total_before:.1f}% retained)")

  A1_train: re-truncated 120 shifted rows (words: ['a', 'addition', 'adds', 'are', 'arithmetic', 'being', 'bit', 'calculation', 'calculations', 'check', 'checks', 'confirms', 'considered', 'definitely', 'does', "don't", 'equal', 'equation', 'errors', 'everything', 'hold', 'holds', 'indeed', 'inverse', "isn't", "it's", 'its', 'letters', 'matches', 'multiplication', 'not', 'obviously', 'or', 'out', "something's", 'sum', 'the', 'to', 'up', 'was', 'which'])
A1_train: 695 -> 282 rows (40.6%) -> ..\CoT_datasets\lexically_cleaned\A1_train.csv
  A1_test: re-truncated 99 shifted rows (words: ['a', 'any', 'are', 'being', 'calculation', 'check', 'checks', 'definitely', "doesn't", 'equation', 'factually', 'here', 'hold', 'holds', "isn't", "it's", 'result', 'they', 'to', 'up', 'was', 'which'])
A1_test: 297 -> 210 rows (70.7%) -> ..\CoT_datasets\lexically_cleaned\A1_test.csv
  A2_train: re-truncated 400 shifted rows (words: ['adding', 'and', 'balances', 'calculation', 'checks', 'claiming', "doesn't"